# TerraDelta error analysis
Read saved probabilities and fully annotated masks. Mock results must not be reported as model quality.


In [ ]:
from pathlib import Path
import sys
import numpy as np
from PIL import Image
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from terradelta.data.dataset import ChangeDataset
from terradelta.metrics import evaluate_predictions
from terradelta.training.validation import prediction_row, ground_truth_row
from terradelta.utils.io import load_config
MANIFEST = ROOT / "data/approved/val.csv"
PROB_DIR = ROOT / "outputs/val_probs"
CONFIG = load_config(ROOT / "configs/inference_baseline.yaml")


In [ ]:
if MANIFEST.is_file() and PROB_DIR.is_dir():
    data = ChangeDataset(MANIFEST)
    predictions, truth = [], []
    for i in range(len(data)):
        sample = data[i]
        if not sample["valid_mask"].all():
            raise ValueError("Error analysis needs complete validation annotations")
        prob = np.load(PROB_DIR / (sample["id"] + ".npy"),allow_pickle=False)
        pred = prediction_row(sample["id"],prob,CONFIG)
        gt = ground_truth_row(sample)
        predictions.append(pred)
        truth.append(gt)
        if i < 4:
            heat = np.stack([prob[1],prob[2],np.zeros_like(prob[1])],axis=-1)
            display(Image.fromarray((np.clip(heat,0,1)*255).astype(np.uint8)))
            print(sample["id"],"pred presence",[bool(pred[k]) for k in ("new_building","tree_removal")])
    print(evaluate_predictions(predictions,truth))
else:
    print("Run forward-only validation on fully reviewed data and save probability maps first.")
